# Notebook Pengujian Model - Muhammad Ivan LSPW

Notebook ini digunakan untuk menguji model yang telah dideploy di cloud dan melakukan request prediksi.

## 1. Persiapan
Pastikan aplikasi Flask (serving.py) sudah berjalan.

In [3]:
import requests
import json

# URL untuk testing
BASE_URL = 'http://localhost:8080'

## 2. Test Health Check

In [4]:
# Test health endpoint
response = requests.get(f'{BASE_URL}/health')
print('Health Check:')
print(f'Status: {response.status_code}')
print(f'Response: {response.json()}')

Health Check:
Status: 200
Response: {'model': 'mock', 'status': 'healthy', 'uptime_seconds': 21.81}


## 3. Contoh Data untuk Prediksi
Berikut adalah contoh data pasien untuk diuji.

In [5]:
test_data = {
    'age': 63,
    'sex': 1,
    'cp': 3,
    'trestbps': 145,
    'chol': 233,
    'fbs': 1,
    'restecg': 0,
    'thalach': 150,
    'exang': 0,
    'oldpeak': 2.3,
    'slope': 0,
    'ca': 0,
    'thal': 1
}

## 4. Melakukan Request Prediksi

In [6]:
try:
    response = requests.post(f'{BASE_URL}/predict', json=test_data)
    if response.status_code == 200:
        result = response.json()
        print('Prediksi berhasil:')
        print(f"Prediction: {result.get('prediction')}")
        print(f"Probability: {result.get('probability')}")
        print(f"Status: {result.get('status')}")
        print(f"Label: {'Positive (Disease)' if result.get('prediction') == 1 else 'Negative (No Disease)'}")
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Prediksi berhasil:
Prediction: 1
Probability: 0.75
Status: success
Label: Positive (Disease)


## 5. Menguji Beberapa Sampel Data

In [7]:
test_samples = [
    {'age': 37, 'sex': 1, 'cp': 2, 'trestbps': 130, 'chol': 250, 'fbs': 0, 'restecg': 1, 'thalach': 187, 'exang': 0, 'oldpeak': 3.5, 'slope': 0, 'ca': 0, 'thal': 2},
    {'age': 41, 'sex': 0, 'cp': 1, 'trestbps': 130, 'chol': 204, 'fbs': 0, 'restecg': 0, 'thalach': 172, 'exang': 0, 'oldpeak': 1.4, 'slope': 2, 'ca': 0, 'thal': 2},
    {'age': 56, 'sex': 1, 'cp': 1, 'trestbps': 120, 'chol': 236, 'fbs': 0, 'restecg': 1, 'thalach': 178, 'exang': 0, 'oldpeak': 0.8, 'slope': 2, 'ca': 0, 'thal': 2}
]

print('Testing Multiple Samples:')
print('='*50)

for i, sample in enumerate(test_samples):
    try:
        response = requests.post(f'{BASE_URL}/predict', json=sample)
        if response.status_code == 200:
            result = response.json()
            label = 'Positive (Disease)' if result.get('prediction') == 1 else 'Negative (No Disease)'
            print(f"Sample {i+1}: Prediction={result.get('prediction')} ({label}), Prob={result.get('probability')}")
        else:
            print(f"Sample {i+1}: Error {response.status_code}")
    except Exception as e:
        print(f"Sample {i+1}: Error - {e}")

Testing Multiple Samples:
Sample 1: Prediction=1 (Positive (Disease)), Prob=0.6
Sample 2: Prediction=0 (Negative (No Disease)), Prob=0.15
Sample 3: Prediction=0 (Negative (No Disease)), Prob=0.25


## 6. Cek Prometheus Metrics

In [8]:
# Test metrics endpoint
response = requests.get(f'{BASE_URL}/metrics')
print('Prometheus Metrics:')
print(f'Status: {response.status_code}')
print('\nMetrics Preview:')
print(response.text[:500])

Prometheus Metrics:
Status: 200

Metrics Preview:
# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 471.0
python_gc_objects_collected_total{generation="1"} 0.0
python_gc_objects_collected_total{generation="2"} 0.0
# HELP python_gc_objects_uncollectable_total Uncollectable object found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncolle
